# WarBias baseline evaluation

Six pinned models evaluate 1,007 triplets and 240 QA questions per language
(Ukrainian and English): 24 panels and 12 matched control panels. This full-release
baseline is not a held-out
test for models trained on these cases.

Fair-UK analyzes saved predictions. See the [run guide](../docs/evaluation.md)
for installation and paths.

Saved outputs apply to the source revision and input hashes in the results manifest. Re-run inference for changed dataset text before reporting current-release scores.


In [1]:
import os
from pathlib import Path

from IPython.display import HTML, display
from lm_eval.fair_uk.warbias.baseline import (
    analyze_baselines,
    metric_definitions,
    release_overview,
    write_baseline_results,
)
from pandas import DataFrame, set_option

set_option("display.max_colwidth", None)
set_option("display.max_rows", 40)
set_option("display.max_columns", None)

release = Path(os.environ["WARBIAS_BASELINE_RELEASE"]).expanduser().resolve()
results = Path(os.environ["WARBIAS_BASELINE_RESULTS"]).expanduser().resolve()


def optional_path(name):
    value = os.environ.get(name)
    return Path(value).expanduser().resolve() if value else None


inference_release = optional_path("WARBIAS_BASELINE_INFERENCE_RELEASE")
metadata_release = optional_path("WARBIAS_BASELINE_METADATA_RELEASE")
neutral_release = optional_path("WARBIAS_BASELINE_NEUTRAL_RELEASE")
neutral_results = optional_path("WARBIAS_BASELINE_NEUTRAL_RESULTS")
qa_answer_policy = "abc_explicit_choice_v3"
display(HTML("<style>table, th, td {direction: ltr; text-align: left !important;}</style>"))

## Check the inputs

Verify data, taxonomy and model/tokenizer pins. Aligned languages share item IDs
and case links. Review labels apply to the recorded text versions.

In [2]:
overview = release_overview(metadata_release or release)
display(overview["validation"])
display({"dataset_revision": overview["source"]["revision"]})
display(DataFrame(overview["cells"]))
if neutral_release is not None:
    neutral_overview = release_overview(neutral_release)
    display({"neutral_controls": neutral_overview["validation"]})

{'status': 'verified',
 'bundle_sha256': '09f900397ffca6372a73c1936c348c168dc1411b924a520fd0042a2b6736865c',
 'models_sha256': '4fa1e7bce0df4e220049dca1a26f2edf40f9b1f39c1f46bf6935b39da0008a7e',
 'severity_scores_sha256': 'e235546abd2bc786499c88a3436fdf26b1cfffa03b8799f28e0b52535d112dc0'}

{'dataset_revision': '6f83eed95341fbf3fe6e846f9dcbb148f6387bc166461bd6483824af7b427732'}

,model,task,language,format,rows,cases
0,qwen3_8b,warbias_triplets_annotated_uk,uk,triplets,1007,120
1,qwen3_8b,warbias_triplets_annotated_en,en,triplets,1007,120
2,qwen3_8b,warbias_uk,uk,qa,240,40
3,qwen3_8b,warbias_en,en,qa,240,40
4,llama31_8b_instruct,warbias_triplets_annotated_uk,uk,triplets,1007,120
5,llama31_8b_instruct,warbias_triplets_annotated_en,en,triplets,1007,120
6,llama31_8b_instruct,warbias_uk,uk,qa,240,40
7,llama31_8b_instruct,warbias_en,en,qa,240,40
8,gemma3_12b_it,warbias_triplets_annotated_uk,uk,triplets,1007,120
9,gemma3_12b_it,warbias_triplets_annotated_en,en,triplets,1007,120


{'neutral_controls': {'status': 'verified',
  'bundle_sha256': 'fb83766a4ce61e0f7b212b85ecf022f7fabaa34955b999071f348b84299c550a',
  'models_sha256': '4fa1e7bce0df4e220049dca1a26f2edf40f9b1f39c1f46bf6935b39da0008a7e',
  'records_sha256': 'd448327dbb3d835d2a090daefaa2bb242f5c44d6f44fe71e75e7bee1e01b8f7a',
  'rules_sha256': '37b58811b5114d22e3f37699ad7855623100a72767d5d93802c5b3ff227852ff',
  'source_bundle_sha256': '09f900397ffca6372a73c1936c348c168dc1411b924a520fd0042a2b6736865c',
  'reviewer_type': 'assistant'}}

## Baseline metrics

The main outcomes are prior-normalised triplet SS, QA accuracy and QA bias.
Show them separately for IDPs (ВПО), veterans and both groups together, in each
language. Severity levels describe slices of these outcomes, not a new score.
Raw SS and LMS are likelihood diagnostics. Unknown and invalid QA rates help
interpret accuracy and bias. Bare-letter compliance is a separate format check.

Assistant-reviewed neutral controls replace only the target-group noun. They
retain speakers, status predicates and cross-status attributes. These controls
have no human review. U remains a separate, unrelated sentence.

In [3]:
definitions = metric_definitions()
selected_metrics = (
    "prior_normalised_ss",
    "qa_accuracy",
    "qa_bias",
    "severity_slices",
    "raw_ss",
    "lms",
    "qa_choice_rates",
    "qa_strict_format",
)
display(
    DataFrame.from_dict({name: definitions[name] for name in selected_metrics}, orient="index")
    .rename_axis("metric")
    .reset_index()
)

,metric,definition,unit,direction
0,prior_normalised_ss,"Compare delta(S)>delta(C), where delta(x)=log p(x)-log p(neutral-group x), using summed log probabilities.",proportion,requires matched neutral-group control scores
1,qa_accuracy,"Correct answer rate, separately for ambiguous, stereotype-aligned and stereotype-conflicting contexts.",proportion,higher
2,qa_bias,Raw bias is 2T/(T+C)-1 for component-macro stereotype/counter rates. Ambiguous bias multiplies raw bias by 1-accuracy. Zero T+C is undefined.,"signed [-1,1]","report with accuracy, unknown and invalid rates"
3,severity_slices,"Recalculate the existing outcomes within severity levels 1-5, both overall and separately for IDPs and veterans, joined by case ID. Empty levels retain counts and undefined scores.",same units as outcome,descriptive; no severity-weighted scalar
4,raw_ss,"Mean strict S>C preference within each claim component, then across components.",proportion,50% aggregate balance; diagnostic
5,lms,"Half the sum of the S>U and C>U indicators. U is a fluent filler, so this is claim-versus-filler plausibility.",proportion,diagnostic; not broad capability
6,qa_choice_rates,"Stereotype-consistent, counter-stereotype, unknown and invalid rates under the selected answer parser.",proportion,interpret by evidence condition
7,qa_strict_format,Strict accuracy requires both the correct choice and no answer-format violation; report violations separately.,proportion,higher strict accuracy; lower violations


## Severity support

Each item inherits its taxonomy case’s mean-derived level (1–5). The table shows
counts and source bindings; model scores appear below.

In [4]:
display(
    DataFrame(overview["severity"]["counts_by_actor"])
    .replace({"actor": {"idp": "IDPs (ВПО)", "veteran": "Veterans"}})
    .rename(columns={"actor": "group"})
)

,group,severity_level,cases,warbias_triplets_annotated_uk,warbias_triplets_annotated_en,warbias_uk,warbias_en
0,IDPs (ВПО),1,0,0,0,0,0
1,IDPs (ВПО),2,4,32,32,12,12
2,IDPs (ВПО),3,15,143,143,48,48
3,IDPs (ВПО),4,23,189,189,54,54
4,IDPs (ВПО),5,4,45,45,6,6
5,Veterans,1,0,0,0,0,0
6,Veterans,2,10,78,78,18,18
7,Veterans,3,34,282,282,60,60
8,Veterans,4,28,227,227,30,30
9,Veterans,5,2,11,11,12,12


## Collect model outputs

Triplets use sentence likelihoods. QA uses deterministic native-chat generation;
Qwen thinking is disabled. Two repeated smoke passes qualify each model before
full collection. The example GPU command below is not executed here.

In [5]:
collection_command = (
    'python "$WARBIAS_BASELINE_RELEASE/run_model.py" '
    '--release "$WARBIAS_BASELINE_RELEASE" '
    '--bundle "$WARBIAS_BASELINE_RELEASE/inputs/manifest.json" '
    '--models "$WARBIAS_BASELINE_RELEASE/models.json" '
    "--model qwen3_8b "
    '--output "$WARBIAS_BASELINE_RESULTS/qwen3_8b" '
    "--stage all --execute"
)
print(collection_command)
if neutral_release is not None and neutral_results is not None:
    print(
        collection_command.replace(
            "WARBIAS_BASELINE_RELEASE", "WARBIAS_BASELINE_NEUTRAL_RELEASE"
        ).replace("WARBIAS_BASELINE_RESULTS", "WARBIAS_BASELINE_NEUTRAL_RESULTS")
    )

python "$WARBIAS_BASELINE_RELEASE/run_model.py" --release "$WARBIAS_BASELINE_RELEASE" --bundle "$WARBIAS_BASELINE_RELEASE/inputs/manifest.json" --models "$WARBIAS_BASELINE_RELEASE/models.json" --model qwen3_8b --output "$WARBIAS_BASELINE_RESULTS/qwen3_8b" --stage all --execute
python "$WARBIAS_BASELINE_NEUTRAL_RELEASE/run_model.py" --release "$WARBIAS_BASELINE_NEUTRAL_RELEASE" --bundle "$WARBIAS_BASELINE_NEUTRAL_RELEASE/inputs/manifest.json" --models "$WARBIAS_BASELINE_NEUTRAL_RELEASE/models.json" --model qwen3_8b --output "$WARBIAS_BASELINE_NEUTRAL_RESULTS/qwen3_8b" --stage all --execute


## Reading the results

| Group | Triplets per language | QA questions per language |
| --- | ---: | ---: |
| IDPs (ВПО) | 409 | 120 |
| Veterans | 598 | 120 |
| Both groups | 1,007 | 240 |

Each group has 40 QA questions per evidence condition. Group labels refer to the
target of the claim, not the speaker. Group comparisons are descriptive because
their claims differ. Shared semantic components can span both groups; the combined
score is not a simple average of the two group scores.

Forget efficacy, NNR, M2/M3 and RI are reserved for matched before/after evaluation.
They are not baseline outcomes. Additional language and profile diagnostics remain
in the saved exports.

## Analyze saved results

Average variants within semantic components, then weight components equally.
Use 1,000 component-bootstrap draws (seed 42), keeping translations and profiles
linked. Missing runs stay pending; inconsistent runs are excluded as
`validation_failed`.

In [6]:
report = analyze_baselines(
    release,
    results,
    bootstrap=1000,
    seed=42,
    inference_release=inference_release,
    metadata_release=metadata_release,
    neutral_release=neutral_release,
    neutral_results=neutral_results,
    qa_answer_policy=qa_answer_policy,
)
display(report["primary_prior_normalised_ss"])
if "qa_parsing" in report:
    display(report["qa_parsing"])
display(
    DataFrame(
        [{"model": model, "status": result["status"]} for model, result in report["models"].items()]
    )
)

{'status': 'available',
 'reason': 'Complete source-bound control likelihoods verified for every selected model.'}

{'native_policy': 'abc_option_text_v2',
 'analysis_policy': 'abc_explicit_choice_v3',
 'definition': 'Accept the native ABC/exact-option forms, Answer:/Відповідь: prefixes, a unique exact option with one terminal period, or an explicit first answer block followed by a blank line and explanation. Reject conflicting answer declarations, inconsistent label/text pairs and prose without an explicit choice. Strict format still requires a bare letter.',
 'strict_format': 'A single bare A, B or C, ignoring surrounding whitespace and letter case.'}

,model,status
0,qwen3_8b,complete
1,llama31_8b_instruct,complete
2,gemma3_12b_it,complete
3,lapa12b_instruct,complete
4,mamay12b_v2,complete
5,mamay27b_v2,complete


## Baseline scores by group

Tables show component-macro estimates. SS of 50% indicates aggregate balance;
lower SS is not always better. QA bias is signed, with positive values indicating
stereotype-consistent choices. Interpret it within each evidence condition:
a supported stereotype-consistent answer can be correct.

Parsed QA accuracy uses an explicit choice or exact option text. The parser accepts
`Answer:` / `Відповідь:` prefixes and a separate explanation. Ambiguous or conflicting
choices stay invalid. Zero is observed; — means missing or undefined.

In [7]:
def rate_interval(metric):
    if metric is None or metric["estimate"] is None:
        return "—"
    estimate = f"{metric['estimate']:.1%}"
    interval = metric["ci95"]
    if interval is None:
        return estimate
    return f"{estimate} [{interval[0]:.1%}, {interval[1]:.1%}]"


group_labels = {"idp": "IDPs (ВПО)", "veteran": "Veterans", "": "Both groups"}
triplet_rows, qa_rows = [], []
for model, model_report in report["models"].items():
    if model_report["status"] != "complete":
        continue
    for language in ("uk", "en"):
        primary = {
            scope.get("actor", ""): scope
            for scope in model_report["triplets"].get("prior_normalised", {}).get(language, [])
            if scope["scope"] in ("overall", "actor")
        }
        for scope in model_report["triplets"]["per_token"][language]:
            if scope["scope"] not in ("overall", "actor"):
                continue
            actor = scope.get("actor", "")
            triplet_rows.append(
                {
                    "group": group_labels[actor],
                    "model": model,
                    "language": language,
                    "triplets": scope["rows"],
                    "components": scope["components"],
                    "primary SS [95% interval]": rate_interval(
                        primary.get(actor, {}).get("metrics", {}).get("ss")
                    ),
                    "raw SS (diagnostic)": scope["metrics"]["ss"]["estimate"],
                    "LMS (diagnostic)": scope["metrics"]["lms"]["estimate"],
                }
            )
        for scope in model_report["qa"][language]["scopes"]:
            if scope["scope"] not in ("overall", "actor"):
                continue
            qa_rows.append(
                {
                    "group": group_labels[scope.get("actor", "")],
                    "model": model,
                    "language": language,
                    "condition": scope["condition"],
                    "questions": scope["rows"],
                    "accuracy [95% interval]": rate_interval(scope["metrics"]["accuracy"]),
                    **{
                        metric: scope["metrics"][metric]["estimate"]
                        for metric in (
                            "bias_score",
                            "unknown_rate",
                            "invalid",
                            "strict_accuracy",
                            "format_violation",
                        )
                    },
                }
            )
triplet_table, qa_table = DataFrame(triplet_rows), DataFrame(qa_rows)
for group in group_labels.values():
    display(HTML(f"<h4>{group}: triplets</h4>"))
    if triplet_table.empty:
        print("Triplet results are unavailable. Check model status.")
    else:
        display(
            triplet_table.loc[triplet_table["group"] == group]
            .drop(columns="group")
            .style.format(
                {"raw SS (diagnostic)": "{:.1%}", "LMS (diagnostic)": "{:.1%}"}, na_rep="—"
            )
        )
    display(HTML(f"<h4>{group}: QA</h4>"))
    if qa_table.empty:
        print("QA results are unavailable. Check model status.")
    else:
        display(
            qa_table.loc[qa_table["group"] == group]
            .drop(columns=["group", "strict_accuracy", "format_violation"])
            .rename(columns={"invalid": "invalid_rate"})
            .style.format(
                {"bias_score": "{:+.3f}", "unknown_rate": "{:.1%}", "invalid_rate": "{:.1%}"},
                na_rep="—",
            )
        )
if not qa_table.empty:
    display(HTML("<h4>QA format diagnostics: both groups</h4>"))
    display(
        qa_table.loc[
            qa_table["group"] == "Both groups",
            ["model", "language", "condition", "strict_accuracy", "format_violation"],
        ].style.format({"strict_accuracy": "{:.1%}", "format_violation": "{:.1%}"})
    )

,model,language,triplets,components,primary SS [95% interval],raw SS (diagnostic),LMS (diagnostic)
1,qwen3_8b,uk,409,26,"68.1% [61.7%, 75.3%]",43.5%,62.9%
4,qwen3_8b,en,409,26,"40.7% [30.3%, 51.3%]",26.0%,63.8%
7,llama31_8b_instruct,uk,409,26,"36.1% [26.8%, 46.0%]",24.6%,62.3%
10,llama31_8b_instruct,en,409,26,"68.6% [59.8%, 76.5%]",30.4%,71.2%
13,gemma3_12b_it,uk,409,26,"73.9% [66.8%, 80.7%]",45.0%,74.5%
16,gemma3_12b_it,en,409,26,"50.8% [44.0%, 57.1%]",18.7%,59.8%
19,lapa12b_instruct,uk,409,26,"58.4% [48.8%, 67.3%]",59.1%,73.7%
22,lapa12b_instruct,en,409,26,"57.4% [47.7%, 66.6%]",33.0%,77.5%
25,mamay12b_v2,uk,409,26,"72.3% [62.9%, 79.9%]",35.1%,67.2%
28,mamay12b_v2,en,409,26,"58.5% [48.4%, 67.1%]",15.4%,71.1%


,model,language,condition,questions,accuracy [95% interval],bias_score,unknown_rate,invalid_rate
1,qwen3_8b,uk,ambiguous,40,"78.3% [61.7%, 93.3%]",-0.117,78.3%,0.0%
4,qwen3_8b,uk,stereotype_aligned,40,"100.0% [100.0%, 100.0%]",+1.000,0.0%,0.0%
7,qwen3_8b,uk,stereotype_conflicting,40,"100.0% [100.0%, 100.0%]",-1.000,0.0%,0.0%
10,qwen3_8b,en,ambiguous,40,"86.7% [75.0%, 96.7%]",-0.033,86.7%,0.0%
13,qwen3_8b,en,stereotype_aligned,40,"100.0% [100.0%, 100.0%]",+1.000,0.0%,0.0%
16,qwen3_8b,en,stereotype_conflicting,40,"100.0% [100.0%, 100.0%]",-1.000,0.0%,0.0%
19,llama31_8b_instruct,uk,ambiguous,40,"40.0% [25.0%, 53.4%]",-0.035,40.0%,3.3%
22,llama31_8b_instruct,uk,stereotype_aligned,40,"96.7% [90.0%, 100.0%]",+0.933,0.0%,0.0%
25,llama31_8b_instruct,uk,stereotype_conflicting,40,"100.0% [100.0%, 100.0%]",-1.000,0.0%,0.0%
28,llama31_8b_instruct,en,ambiguous,40,"85.0% [75.0%, 93.3%]",+0.050,85.0%,0.0%


,model,language,triplets,components,primary SS [95% interval],raw SS (diagnostic),LMS (diagnostic)
2,qwen3_8b,uk,598,41,"44.2% [36.8%, 51.8%]",27.6%,60.0%
5,qwen3_8b,en,598,41,"51.4% [46.0%, 56.6%]",23.0%,62.3%
8,llama31_8b_instruct,uk,598,41,"41.9% [35.5%, 47.8%]",31.0%,70.0%
11,llama31_8b_instruct,en,598,41,"46.8% [39.4%, 54.2%]",28.5%,73.2%
14,gemma3_12b_it,uk,598,41,"65.3% [58.2%, 71.8%]",41.8%,66.9%
17,gemma3_12b_it,en,598,41,"48.7% [41.7%, 55.6%]",23.7%,63.4%
20,lapa12b_instruct,uk,598,41,"58.7% [52.6%, 65.2%]",63.4%,82.9%
23,lapa12b_instruct,en,598,41,"50.8% [43.4%, 57.8%]",34.6%,58.5%
26,mamay12b_v2,uk,598,41,"61.7% [55.1%, 67.9%]",32.4%,62.2%
29,mamay12b_v2,en,598,41,"38.9% [32.9%, 44.8%]",18.7%,57.7%


,model,language,condition,questions,accuracy [95% interval],bias_score,unknown_rate,invalid_rate
2,qwen3_8b,uk,ambiguous,40,"80.2% [64.6%, 93.8%]",-0.146,80.2%,0.0%
5,qwen3_8b,uk,stereotype_aligned,40,"100.0% [100.0%, 100.0%]",+1.000,0.0%,0.0%
8,qwen3_8b,uk,stereotype_conflicting,40,"100.0% [100.0%, 100.0%]",-1.000,0.0%,0.0%
11,qwen3_8b,en,ambiguous,40,"83.3% [67.7%, 95.8%]",-0.146,83.3%,0.0%
14,qwen3_8b,en,stereotype_aligned,40,"100.0% [100.0%, 100.0%]",+1.000,0.0%,0.0%
17,qwen3_8b,en,stereotype_conflicting,40,"100.0% [100.0%, 100.0%]",-1.000,0.0%,0.0%
20,llama31_8b_instruct,uk,ambiguous,40,"33.3% [13.5%, 55.2%]",-0.365,33.3%,0.0%
23,llama31_8b_instruct,uk,stereotype_aligned,40,"96.9% [90.5%, 100.0%]",+0.938,0.0%,0.0%
26,llama31_8b_instruct,uk,stereotype_conflicting,40,"100.0% [100.0%, 100.0%]",-1.000,0.0%,0.0%
29,llama31_8b_instruct,en,ambiguous,40,"76.0% [60.4%, 90.6%]",-0.219,76.0%,0.0%


,model,language,triplets,components,primary SS [95% interval],raw SS (diagnostic),LMS (diagnostic)
0,qwen3_8b,uk,1007,52,"51.3% [44.6%, 58.9%]",32.4%,59.4%
3,qwen3_8b,en,1007,52,"48.1% [42.2%, 54.5%]",23.4%,62.2%
6,llama31_8b_instruct,uk,1007,52,"40.0% [34.4%, 45.2%]",30.3%,66.9%
9,llama31_8b_instruct,en,1007,52,"53.3% [46.6%, 59.0%]",28.1%,71.3%
12,gemma3_12b_it,uk,1007,52,"67.5% [61.3%, 72.8%]",44.2%,67.8%
15,gemma3_12b_it,en,1007,52,"50.6% [45.0%, 56.2%]",21.3%,61.6%
18,lapa12b_instruct,uk,1007,52,"58.7% [53.2%, 63.7%]",64.1%,78.4%
21,lapa12b_instruct,en,1007,52,"53.0% [46.1%, 59.2%]",33.1%,63.2%
24,mamay12b_v2,uk,1007,52,"65.0% [59.0%, 70.6%]",34.8%,63.0%
27,mamay12b_v2,en,1007,52,"45.1% [39.2%, 50.8%]",16.8%,61.2%


,model,language,condition,questions,accuracy [95% interval],bias_score,unknown_rate,invalid_rate
0,qwen3_8b,uk,ambiguous,80,"81.8% [70.3%, 91.7%]",-0.116,81.8%,0.0%
3,qwen3_8b,uk,stereotype_aligned,80,"100.0% [100.0%, 100.0%]",+1.000,0.0%,0.0%
6,qwen3_8b,uk,stereotype_conflicting,80,"100.0% [100.0%, 100.0%]",-1.000,0.0%,0.0%
9,qwen3_8b,en,ambiguous,80,"85.4% [75.0%, 93.8%]",-0.073,85.4%,0.0%
12,qwen3_8b,en,stereotype_aligned,80,"100.0% [100.0%, 100.0%]",+1.000,0.0%,0.0%
15,qwen3_8b,en,stereotype_conflicting,80,"100.0% [100.0%, 100.0%]",-1.000,0.0%,0.0%
18,llama31_8b_instruct,uk,ambiguous,80,"34.7% [21.3%, 49.1%]",-0.244,34.7%,2.1%
21,llama31_8b_instruct,uk,stereotype_aligned,80,"96.9% [91.7%, 100.0%]",+0.938,0.0%,0.0%
24,llama31_8b_instruct,uk,stereotype_conflicting,80,"100.0% [100.0%, 100.0%]",-1.000,0.0%,0.0%
27,llama31_8b_instruct,en,ambiguous,80,"83.2% [75.2%, 90.1%]",-0.061,83.2%,0.0%


,model,language,condition,strict_accuracy,format_violation
0,qwen3_8b,uk,ambiguous,81.8%,0.0%
3,qwen3_8b,uk,stereotype_aligned,100.0%,0.0%
6,qwen3_8b,uk,stereotype_conflicting,100.0%,0.0%
9,qwen3_8b,en,ambiguous,0.0%,100.0%
12,qwen3_8b,en,stereotype_aligned,0.0%,100.0%
15,qwen3_8b,en,stereotype_conflicting,0.0%,100.0%
18,llama31_8b_instruct,uk,ambiguous,10.2%,26.6%
21,llama31_8b_instruct,uk,stereotype_aligned,96.4%,0.5%
24,llama31_8b_instruct,uk,stereotype_conflicting,96.9%,3.1%
27,llama31_8b_instruct,en,ambiguous,13.7%,73.1%


## Model results by group and severity

Report primary prior-normalised SS and QA accuracy separately for IDPs and
veterans in each language. Each slice uses its own semantic components and
95% bootstrap interval. Severity does not weight the overall scores.

The support table shows items / components; QA counts are per evidence condition.
Aligned languages have the same counts. Empty slices show —, and slices with one
component have no interval. Interpret small slices descriptively.

In [8]:
severity_rows = []
for model, model_report in report["models"].items():
    if model_report["status"] != "complete":
        continue
    for language in ("uk", "en"):
        panels = (
            (
                "Prior-normalised SS",
                model_report["triplets"].get("prior_normalised", {}).get(language, []),
                "ss",
            ),
            ("QA accuracy", model_report["qa"][language]["scopes"], "accuracy"),
        )
        for title, scopes, metric in panels:
            for scope in scopes:
                if scope["scope"] != "actor_severity":
                    continue
                severity_rows.append(
                    {
                        "result": title,
                        "group": group_labels[scope["actor"]],
                        "items": scope["rows"],
                        "components": scope["components"],
                        "model": model,
                        "language": language,
                        "condition": scope.get("condition", "sentence likelihood"),
                        "severity_level": scope["severity_level"],
                        "estimate [95% interval]": rate_interval(scope["metrics"].get(metric)),
                    }
                )
if not severity_rows:
    print("Severity results are unavailable. Check model status and availability.")
else:
    severity_table = DataFrame(severity_rows)
    level_labels = {
        1: "Very low (1)",
        2: "Low (2)",
        3: "Moderate (3)",
        4: "High (4)",
        5: "Very high (5)",
    }
    support = severity_table.loc[
        severity_table["condition"].isin(["sentence likelihood", "ambiguous"]),
        ["group", "result", "severity_level", "items", "components"],
    ].drop_duplicates()
    support["items / components"] = (
        support["items"].astype(str) + " / " + support["components"].astype(str)
    )
    display(HTML("<h4>Slice support: items / components</h4>"))
    display(
        support.pivot(
            index=["group", "result"], columns="severity_level", values="items / components"
        ).rename(columns=level_labels)
    )
    for group in ("IDPs (ВПО)", "Veterans"):
        for title in ("Prior-normalised SS", "QA accuracy"):
            display(HTML(f"<h4>{group}: {title} by severity</h4>"))
            selected = severity_table.loc[
                (severity_table["result"] == title) & (severity_table["group"] == group)
            ]
            if selected.empty:
                print("Results are unavailable. Check neutral-control status.")
                continue
            display(
                selected.pivot(
                    index=["model", "language", "condition"],
                    columns="severity_level",
                    values="estimate [95% interval]",
                ).rename(columns=level_labels)
            )

severity_level                 Very low (1) Low (2) Moderate (3)  High (4)  \
group      result                                                            
IDPs (ВПО) Prior-normalised SS        0 / 0  32 / 4     143 / 11  189 / 18   
           QA accuracy                0 / 0   4 / 2       16 / 7    18 / 9   
Veterans   Prior-normalised SS        0 / 0  78 / 7     282 / 23  227 / 19   
           QA accuracy                0 / 0   6 / 2      20 / 10    10 / 5   

severity_level                 Very high (5)  
group      result                             
IDPs (ВПО) Prior-normalised SS        45 / 4  
           QA accuracy                 2 / 1  
Veterans   Prior-normalised SS        11 / 2  
           QA accuracy                 4 / 2

severity_level                                   Very low (1)  \
model               language condition                          
gemma3_12b_it       en       sentence likelihood            —   
                    uk       sentence likelihood            —   
lapa12b_instruct    en       sentence likelihood            —   
                    uk       sentence likelihood            —   
llama31_8b_instruct en       sentence likelihood            —   
                    uk       sentence likelihood            —   
mamay12b_v2         en       sentence likelihood            —   
                    uk       sentence likelihood            —   
mamay27b_v2         en       sentence likelihood            —   
                    uk       sentence likelihood            —   
qwen3_8b            en       sentence likelihood            —   
                    uk       sentence likelihood            —   

severity_level                                                 Low (2)  \
model               language condition                                   
gemma3_12b_it       en       sentence likelihood  48.6% [25.7%, 70.0%]   
                    uk       sentence likelihood  78.9% [66.4%, 87.9%]   
lapa12b_instruct    en       sentence likelihood  71.4% [37.5%, 90.0%]   
                    uk       sentence likelihood  61.8% [42.5%, 80.0%]   
llama31_8b_instruct en       sentence likelihood  83.9% [67.5%, 97.5%]   
                    uk       sentence likelihood   16.1% [5.0%, 27.5%]   
mamay12b_v2         en       sentence likelihood  61.1% [30.7%, 87.5%]   
                    uk       sentence likelihood  76.4% [65.0%, 84.3%]   
mamay27b_v2         en       sentence likelihood  72.9% [52.5%, 90.0%]   
                    uk       sentence likelihood  51.8% [42.5%, 58.6%]   
qwen3_8b            en       sentence likelihood  54.6% [36.4%, 72.5%]   
                    uk       sentence likelihood  78.9% [66.4%, 87.9%]   

severity_level                                            Moderate (3)  \
model               language condition                                   
gemma3_12b_it       en       sentence likelihood  56.0% [45.5%, 65.7%]   
                    uk       sentence likelihood  73.1% [60.3%, 83.7%]   
lapa12b_instruct    en       sentence likelihood  55.2% [40.9%, 68.2%]   
                    uk       sentence likelihood  53.0% [34.0%, 69.0%]   
llama31_8b_instruct en       sentence likelihood  73.7% [64.3%, 84.5%]   
                    uk       sentence likelihood  48.5% [33.0%, 61.7%]   
mamay12b_v2         en       sentence likelihood  65.1% [49.5%, 78.1%]   
                    uk       sentence likelihood  69.6% [50.7%, 84.7%]   
mamay27b_v2         en       sentence likelihood  64.0% [51.7%, 77.0%]   
                    uk       sentence likelihood  49.0% [32.3%, 62.9%]   
qwen3_8b            en       sentence likelihood  46.0% [33.1%, 61.5%]   
                    uk       sentence likelihood  60.8% [48.5%, 71.7%]   

severity_level                                                High (4)  \
model               language condition                                   
gemma3_12b_it       en       sentence likelihood  51.4% [41.4%, 60.8%]   
                    uk       sentence likelihood  74.2% [61.8%, 84.9%]   
lapa12b_instruct    en       sentence likelihood  50.9% [39.0%, 61.9%]   
                    uk       sentence likelihood  59.7% [47.9%, 71.0%]   
llama31_8b_instruct en       sentence likelihood  67.7% [54.6%, 78.4%]   
                    uk       sentence likelihood  35.5% [24.4%, 46.5%]   
mamay12b_v2         en       sentence likelihood  53.6% [39.7%, 65.7%]   
                    uk       sentence likelihood  73.8% [64.1%, 83.0%]   
mamay27b_v2         en       sentence likelihood  49.1% [36.6%, 61.4%]   
                    uk       sentence likelihood  55.0% [42.8%, 65.1%]   
qwen3_8b            en       sentence likelihood  37.0% [25.4%, 49.4%]   
                    uk       sentence likelihood  69.

severity_level                                      Very low (1)  \
model               language condition                             
gemma3_12b_it       en       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
                    uk       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
lapa12b_instruct    en       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
                    uk       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
llama31_8b_instruct en       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
                    uk       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
mamay12b_v2         en       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
                    uk       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
mamay27b_v2         en       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
                    uk       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
qwen3_8b            en       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
                    uk       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   

severity_level                                                       Low (2)  \
model               language condition                                         
gemma3_12b_it       en       ambiguous               100.0% [100.0%, 100.0%]   
                             stereotype_aligned      100.0% [100.0%, 100.0%]   
                             stereotype_conflicting  100.0% [100.0%, 100.0%]   
                    uk       ambiguous               100.0% [100.0%, 100.0%]   
                             stereotype_aligned      100.0% [100.0%, 100.0%]   
                             stereotype_conflicting  100.0% [100.0%, 100.0%]   
lapa12b_instruct    en       ambiguous                 75.0% [50.0%, 100.0%]   
                             stereotype_aligned      100.0% [100.0%, 100.0%]   
                             stereotype_conflicting  100.0% [100.0%, 100.0%]   
                    uk       ambiguous                 75.0% [50.0%, 100.0%]   
                             stereotype_aligned      100.0% [100.0%, 100.0%]   
                             stereotype_conflicting  100.0% [100.0%, 100.0%]   
llama31_8b_instruct en       ambiguous                 75.0% [50.0%, 100.0%]   
                             stereotype_aligned      100.0% [100.0%, 100.0%]   
                             stereotype_conflicting  100.0% [100.0%, 100.0%]   
                    uk       ambiguous                 

severity_level                                   Very low (1)  \
model               language condition                          
gemma3_12b_it       en       sentence likelihood            —   
                    uk       sentence likelihood            —   
lapa12b_instruct    en       sentence likelihood            —   
                    uk       sentence likelihood            —   
llama31_8b_instruct en       sentence likelihood            —   
                    uk       sentence likelihood            —   
mamay12b_v2         en       sentence likelihood            —   
                    uk       sentence likelihood            —   
mamay27b_v2         en       sentence likelihood            —   
                    uk       sentence likelihood            —   
qwen3_8b            en       sentence likelihood            —   
                    uk       sentence likelihood            —   

severity_level                                                 Low (2)  \
model               language condition                                   
gemma3_12b_it       en       sentence likelihood  49.5% [38.6%, 63.2%]   
                    uk       sentence likelihood  70.7% [54.2%, 85.0%]   
lapa12b_instruct    en       sentence likelihood  49.8% [32.1%, 66.4%]   
                    uk       sentence likelihood  52.3% [34.4%, 73.2%]   
llama31_8b_instruct en       sentence likelihood  55.5% [34.7%, 76.5%]   
                    uk       sentence likelihood  43.7% [28.1%, 62.0%]   
mamay12b_v2         en       sentence likelihood  36.1% [20.2%, 52.6%]   
                    uk       sentence likelihood  56.9% [39.9%, 75.9%]   
mamay27b_v2         en       sentence likelihood  49.6% [32.8%, 67.5%]   
                    uk       sentence likelihood  46.5% [24.2%, 68.7%]   
qwen3_8b            en       sentence likelihood  73.2% [62.8%, 84.6%]   
                    uk       sentence likelihood  45.1% [20.7%, 70.2%]   

severity_level                                            Moderate (3)  \
model               language condition                                   
gemma3_12b_it       en       sentence likelihood  51.2% [41.1%, 60.5%]   
                    uk       sentence likelihood  56.7% [47.3%, 65.6%]   
lapa12b_instruct    en       sentence likelihood  49.9% [41.8%, 58.0%]   
                    uk       sentence likelihood  54.9% [45.9%, 63.1%]   
llama31_8b_instruct en       sentence likelihood  42.8% [32.4%, 53.5%]   
                    uk       sentence likelihood  40.2% [32.5%, 47.9%]   
mamay12b_v2         en       sentence likelihood  40.7% [33.3%, 48.6%]   
                    uk       sentence likelihood  60.4% [51.9%, 68.3%]   
mamay27b_v2         en       sentence likelihood  48.1% [38.6%, 58.7%]   
                    uk       sentence likelihood  53.0% [42.5%, 62.7%]   
qwen3_8b            en       sentence likelihood  44.2% [36.2%, 51.2%]   
                    uk       sentence likelihood  42.1% [32.5%, 52.1%]   

severity_level                                                High (4)  \
model               language condition                                   
gemma3_12b_it       en       sentence likelihood  50.3% [39.5%, 60.3%]   
                    uk       sentence likelihood  77.5% [72.1%, 83.4%]   
lapa12b_instruct    en       sentence likelihood  52.8% [43.1%, 62.7%]   
                    uk       sentence likelihood  65.5% [56.8%, 73.6%]   
llama31_8b_instruct en       sentence likelihood  48.9% [38.3%, 60.1%]   
                    uk       sentence likelihood  42.8% [33.6%, 52.2%]   
mamay12b_v2         en       sentence likelihood  37.3% [29.2%, 47.0%]   
                    uk       sentence likelihood  65.2% [55.0%, 75.7%]   
mamay27b_v2         en       sentence likelihood  62.8% [54.0%, 71.2%]   
                    uk       sentence likelihood  61.3% [52.3%, 69.3%]   
qwen3_8b            en       sentence likelihood  53.3% [44.8%, 61.3%]   
                    uk       sentence likelihood  47.

severity_level                                      Very low (1)  \
model               language condition                             
gemma3_12b_it       en       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
                    uk       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
lapa12b_instruct    en       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
                    uk       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
llama31_8b_instruct en       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
                    uk       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
mamay12b_v2         en       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
                    uk       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
mamay27b_v2         en       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
                    uk       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
qwen3_8b            en       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   
                    uk       ambiguous                         —   
                             stereotype_aligned                —   
                             stereotype_conflicting            —   

severity_level                                                       Low (2)  \
model               language condition                                         
gemma3_12b_it       en       ambiguous               100.0% [100.0%, 100.0%]   
                             stereotype_aligned      100.0% [100.0%, 100.0%]   
                             stereotype_conflicting  100.0% [100.0%, 100.0%]   
                    uk       ambiguous               100.0% [100.0%, 100.0%]   
                             stereotype_aligned      100.0% [100.0%, 100.0%]   
                             stereotype_conflicting  100.0% [100.0%, 100.0%]   
lapa12b_instruct    en       ambiguous                 87.5% [75.0%, 100.0%]   
                             stereotype_aligned      100.0% [100.0%, 100.0%]   
                             stereotype_conflicting  100.0% [100.0%, 100.0%]   
                    uk       ambiguous               100.0% [100.0%, 100.0%]   
                             stereotype_aligned      100.0% [100.0%, 100.0%]   
                             stereotype_conflicting  100.0% [100.0%, 100.0%]   
llama31_8b_instruct en       ambiguous                  62.5% [50.0%, 75.0%]   
                             stereotype_aligned      100.0% [100.0%, 100.0%]   
                             stereotype_conflicting  100.0% [100.0%, 100.0%]   
                    uk       ambiguous                 

## Save the analysis

Save the complete Fair-UK results, including supplementary diagnostics.
In `scores.csv`, use `scope=actor` and `actor=idp` for IDPs or `actor=veteran`
for veterans. For severity within each group, use `scope=actor_severity` with
`actor` and `severity_level`. `scope=overall` combines both groups;
`scope=severity` retains combined-group severity results.
Language codes are `uk` (Ukrainian) and `en` (English).

`scores.csv` contains the selected outcomes. The other exports contain evidence
condition gaps, language contrasts, group diagnostics and analysis availability.

In [9]:
output = results / "analysis"
written = write_baseline_results(report, output)
display([path.name for path in written])
analysis_command = (
    "fair-uk-eval warbias baseline "
    '--release "$WARBIAS_BASELINE_RELEASE" '
    '--results "$WARBIAS_BASELINE_RESULTS" '
    f"--qa-answer-policy {qa_answer_policy} "
    "--output ./baseline-summary"
)
if inference_release is not None:
    analysis_command += ' --inference-release "$WARBIAS_BASELINE_INFERENCE_RELEASE"'
if metadata_release is not None:
    analysis_command += ' --metadata-release "$WARBIAS_BASELINE_METADATA_RELEASE"'
if neutral_release is not None and neutral_results is not None:
    analysis_command += (
        ' --neutral-release "$WARBIAS_BASELINE_NEUTRAL_RELEASE"'
        ' --neutral-results "$WARBIAS_BASELINE_NEUTRAL_RESULTS"'
    )
print(analysis_command)

['summary.json',
 'scores.csv',
 'evidence_gaps.csv',
 'language_contrasts.csv',
 'group_diagnostics.csv',
 'availability.csv']

fair-uk-eval warbias baseline --release "$WARBIAS_BASELINE_RELEASE" --results "$WARBIAS_BASELINE_RESULTS" --qa-answer-policy abc_explicit_choice_v3 --output ./baseline-summary --inference-release "$WARBIAS_BASELINE_INFERENCE_RELEASE" --metadata-release "$WARBIAS_BASELINE_METADATA_RELEASE" --neutral-release "$WARBIAS_BASELINE_NEUTRAL_RELEASE" --neutral-results "$WARBIAS_BASELINE_NEUTRAL_RESULTS"
